# ECOS risk-free (CD91) and KRX delisting history — data availability probe

**Goal:** Resolve two remaining data decisions.
1. **ECOS CD91 risk-free** — confirm the start date and unit of the 91-day CD rate (used to build the risk-free series for excess returns).
2. **KRX delisting history** — confirm back-coverage and completeness of delisted tickers and delisting dates (for delisting treatment / survivorship-bias logging and panel completeness).

The breakpoint universe is the full KOSPI+KOSDAQ market, so delisting history is needed for both markets.

**Install:** `pip install PublicDataReader finance-datareader`

**ECOS key:** ECOS requires its own authentication key (a separate system from OpenDART and KRX). Without it, skip Section 2 and re-run after issuance.
- ecos.bok.or.kr/api -> apply for a key -> identity verification -> enter email/details -> receive the key by email (immediate).
- Add the received key to the project-root `.env` as a single line `ECOS_API_KEY=...` and restart the kernel.

**Note:** Section 1 (delisting history) uses FinanceDataReader and needs no key — it can be run immediately.

In [2]:
# [0] env + imports
import os
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
import FinanceDataReader as fdr

# ECOS only when a key is available
ecos = None
try:
    from PublicDataReader import Ecos
    ECOS_KEY = os.environ.get("ECOS_API_KEY")
    if ECOS_KEY:
        ecos = Ecos(ECOS_KEY)
        print("ECOS: ready")
    else:
        print("ECOS: ECOS_API_KEY not set — skipping Section 2 (add key to .env and restart kernel)")
except ImportError:
    print("ECOS: PublicDataReader not installed — pip install PublicDataReader")

print("FinanceDataReader:", getattr(fdr, "__version__", "?"))

def pick_col(df, keys):
    """Return the first column whose name contains any of keys (case-insensitive)."""
    for c in df.columns:
        cu = str(c).upper()
        if any(k.upper() in cu for k in keys):
            return c
    return None

ECOS: 준비 완료
FinanceDataReader: 0.9.202


## Section 1 — KRX delisting history (FinanceDataReader, no key required)

`StockListing('KRX-DELISTING')` returns all KRX delisted tickers (going back to the 1950s). This is the basis for delisting-treatment / survivorship-bias logging and panel completeness.

In [3]:
# [1] all delisted tickers + schema check
delisted = fdr.StockListing("KRX-DELISTING")
print("total delisted tickers:", len(delisted))
print("columns:", list(delisted.columns))
delisted.head()

폐지 종목 총수: 4146
컬럼: ['Symbol', 'Name', 'Market', 'SecuGroup', 'Kind', 'ListingDate', 'DelistingDate', 'Reason', 'ArrantEnforceDate', 'ArrantEndDate', 'Industry', 'ParValue', 'ListingShares', 'ToSymbol', 'ToName']


,Symbol,Name,Market,SecuGroup,Kind,ListingDate,DelistingDate,Reason,ArrantEnforceDate,ArrantEndDate,Industry,ParValue,ListingShares,ToSymbol,ToName
0,028740,경성전기,KOSPI,주권,NaN,1956-03-03,1961-06-30,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN
1,028730,남선전기,KOSPI,주권,NaN,1956-03-03,1961-06-30,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN
2,034380,조선맥주,KOSPI,주권,NaN,1956-10-01,1960-11-26,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN
3,028720,수도극장,KOSPI,주권,NaN,1957-07-01,1960-11-21,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN
4,028750,한국운수,KOSPI,주권,NaN,1956-03-03,1962-01-04,상장폐지유예기간종료,NaT,NaT,NaN,NaN,NaN,NaN,NaN


In [4]:
# [2] delisting-history coverage analysis
d = delisted.copy()
dcol = pick_col(d, ["DelistingDate", "폐지"])
lcol = pick_col(d, ["ListingDate", "상장일"])
mcol = pick_col(d, ["Market", "시장"])
scol = pick_col(d, ["SecuGroup", "Kind", "증권"])

d[dcol] = pd.to_datetime(d[dcol], errors="coerce")
print("delisting-date column:", dcol)
print("delisting-date range:", d[dcol].min(), "~", d[dcol].max())
print("delisting-date missing:", int(d[dcol].isna().sum()), "/", len(d))

if mcol:
    print()
    print("delisted tickers by market:")
    print(d[mcol].value_counts())

if scol:
    print()
    print(f"by {scol} (top 10):")
    print(d[scol].value_counts().head(10))

print()
print("delistings by decade (back-coverage check):")
d["decade"] = (d[dcol].dt.year // 10) * 10
print(d.groupby("decade").size())

폐지일 컬럼: DelistingDate
폐지일 범위: 1960-11-21 00:00:00 ~ 2026-06-30 00:00:00
폐지일 결측: 0 / 4146

시장별 폐지 종목수:
Market
KOSPI     2124
KOSDAQ    1903
KONEX      119
Name: count, dtype: int64

SecuGroup별(상위 10):
SecuGroup
주권         2086
신주인수권증서     854
수익증권        779
투자회사        176
신주인수권증권     160
선박투자회사       55
부동산투자회사      18
외국주권         12
주식예탁증권        6
Name: count, dtype: int64

폐지 10년단위 추이(백커버리지 확인):
decade
1960       6
1970      35
1980      79
1990     476
2000    1334
2010    1200
2020    1016
dtype: int64


## Panel completeness — currently listed ∪ delisted = survivorship-bias-free universe

FDR's `KRX-DESC` returns **currently listed only**. A survivorship-bias-free panel must be the union of (currently listed ∪ delisted). Since the breakpoint universe is KOSPI+KOSDAQ, both markets are included.

In [5]:
# [3] currently-listed size + all-time universe size estimate
listed = fdr.StockListing("KRX-DESC")
print("currently listed:", len(listed))
print("columns:", list(listed.columns))
lmcol = pick_col(listed, ["Market", "시장"])
if lmcol:
    print()
    print("currently listed by market:")
    print(listed[lmcol].value_counts())
print()
print(f"all-time universe size ~ {len(listed)} (listed) + {len(delisted)} (delisted) = {len(listed)+len(delisted)}")
print("-> survivorship-bias-free panel = this union. Initial version filters to KOSPI+KOSDAQ common stock (excluding preferred shares, ETFs, REITs).")

현재 상장: 2876
컬럼: ['Code', 'Name', 'Market', 'Sector', 'Industry', 'Products', 'ListingDate', 'SettleMonth', 'Representative', 'HomePage', 'Region']

현재상장 시장별:
Market
KOSDAQ           1774
KOSPI             945
KONEX             107
KOSDAQ GLOBAL      50
Name: count, dtype: int64

All-time 유니버스 규모 ≈ 2876(현재) + 4146(폐지) = 7022
→ 생존편향 없는 패널 = 이 합집합. v0.1은 KOSPI+KOSDAQ 보통주로 필터(우선주·ETF·리츠 제외).


## Section 2 — ECOS CD91 risk-free (PublicDataReader, key required)

**Auto-discover** the statistics table code and the CD (91-day) item code and its available date range (minimizing hard-coding). The market-rate (daily) table is usually `817Y002`.

In [8]:
# [4+5] discover CD91 item code + available date range (auto-adapts to columns + diagnostics)
STAT = "817Y002"   # market rate (daily). If this line errors, report the message.

def col_like(df, *keys):
    for c in df.columns:
        s = str(c).upper()
        if any(k.upper() in s for k in keys):
            return c
    return None

items = ecos.get_statistic_item_list(통계표코드=STAT)
print("=== item_list columns ===")
print(list(items.columns), "| rows:", len(items))
print()

namecol  = col_like(items, "항목명", "ITEM_NAME", "CONT_NAME", "명", "NAME")
codecol  = col_like(items, "항목코드", "ITEM_CODE", "CONT_CODE", "코드", "CODE")
startcol = col_like(items, "수록시작", "시작", "START")
endcol   = col_like(items, "수록종료", "종료", "END")
cyclecol = col_like(items, "주기", "CYCLE")
print("detected: name=", namecol, "| code=", codecol, "| start=", startcol, "| end=", endcol)
print()

CD91_CODE = None
if namecol is None or codecol is None:
    print("auto-detection of item name/code failed. Inspect the full table below to find the CD91 row (also report the column names):")
    print(items.head(30).to_string())
else:
    cd = items[items[namecol].astype(str).str.contains("CD", na=False)]
    show = [c for c in [codecol, namecol, cyclecol, startcol, endcol] if c]
    print("=== CD-related items ===")
    print(cd[show].to_string(index=False) if len(cd) else "(no CD items)")
    cd91 = cd[cd[namecol].astype(str).str.contains("91", na=False)] if len(cd) else cd
    src = cd91 if len(cd91) else cd
    if len(src):
        CD91_CODE = str(src.iloc[0][codecol])
    print()
    print(">>> CD91_CODE =", CD91_CODE)
    if startcol and len(src):
        print(">>> CD91 available range:", src.iloc[0][startcol], "~", (src.iloc[0][endcol] if endcol else "?"))

=== item_list 컬럼 ===
['통계표코드', '통계명', '항목그룹코드', '항목그룹명', '통계항목코드', '통계항목명', '상위통계항목코드', '상위통계항목명', '시점', '수록시작일자', '수록종료일자', '자료수', '단위', '가중치'] | 행수: 27

탐지: name= 통계명 | code= 통계표코드 | start= 수록시작일자 | end= 수록종료일자

=== CD 관련 항목 ===
(CD 항목 없음)

>>> CD91_CODE = None


In [9]:
# [6] CD91 sample query — confirm unit and actual start (secondary check)
s = ecos.get_statistic_search(
    통계표코드=STAT, 주기="D",
    검색시작일자="19900101", 검색종료일자="19951231",
    통계항목코드1=CD91_CODE)
print("=== search columns ===")
print(list(s.columns) if s is not None else "returned None", "| rows:", 0 if s is None else len(s))
print()
if s is not None and len(s):
    print(s.head(8).to_string(index=False))
    tcol = col_like(s, "시점", "TIME")
    ucol = col_like(s, "단위", "UNIT")
    if tcol: print("earliest timestamp in this window:", s[tcol].min())
    if ucol: print("unit:", s[ucol].iloc[0])
else:
    print("no data for 1990-95 -> raise 검색시작일자 to 2000 and retry (or use the available-range start above).")
print()
print("Risk-free conversion: CD91 annualized % -> monthly = (1+r/100)**(1/12)-1  or  r/100/12.")

=== search 컬럼 ===
['통계표코드', '통계명', '통계항목코드1', '통계항목명1', '통계항목코드2', '통계항목명2', '통계항목코드3', '통계항목명3', '통계항목코드4', '통계항목명4', '단위', 'WGT', '시점', '값'] | 행수: 1441

  통계표코드               통계명   통계항목코드1        통계항목명1 통계항목코드2 통계항목명2 통계항목코드3 통계항목명3 통계항목코드4 통계항목명4 단위  WGT       시점     값
817Y002 1.3.2.1. 시장금리(일별) 010101000 콜금리(1일, 전체거래)    None   None    None   None    None   None 연% None 19950103 13.06
817Y002 1.3.2.1. 시장금리(일별) 010300000  회사채(3년, AA-)    None   None    None   None    None   None 연% None 19950103 14.25
817Y002 1.3.2.1. 시장금리(일별) 010400001      통안증권(1년)    None   None    None   None    None   None 연% None 19950103 13.85
817Y002 1.3.2.1. 시장금리(일별) 010502000       CD(91일)    None   None    None   None    None   None 연% None 19950103  14.7
817Y002 1.3.2.1. 시장금리(일별) 010503000       CP(91일)    None   None    None   None    None   None 연% None 19950103 15.37
817Y002 1.3.2.1. 시장금리(일별) 010101000 콜금리(1일, 전체거래)    None   None    None   None    None   None 연% None 19950104 11.63
817Y002 1.3.2.1. 시장

## Section 3 — Summary judgment

The two decisions this probe closes:

- **Risk-free start date (ECOS CD91):** the start of the available range in Section 2 is the floor of the risk-free sample. Compare against the B/M cross-section (2003) and market cap (1995) to see **which one binds the overall sample floor**. (If CD91 starts in the 1990s, risk-free is not the binding constraint -> the floor is B/M at 2003.)
- **Delisting-history back-coverage (KRX-DELISTING):** the delisting-date range and per-market distribution in Section 1 set the range over which survivorship-bias logging is possible. If delisting-date missingness is low and both KOSPI and KOSDAQ are captured -> a survivorship-bias-free panel can be constructed.

**Next:** record both start dates, finalize the data-availability decisions, and begin ETL design.
Sample-floor candidates: Mkt-RF / SMB = market cap from 1995, **HML = B/M from 2003 (likely binding)**, risk-free = (confirmed by this probe).